In [ ]:
import numpy as np
import itertools

STUDENT_ID = '2021030028'

In [ ]:
class User():
    def __init__(self, C, p_bg, p_gb, p_packet, initial_state):
        self.C = C
        self.queue = 0
        self.p_bg = p_bg
        self.p_gb = p_gb
        self.p_packet = p_packet
        self.state = initial_state

    def serve(self, N):
        rate = N if self.state == 'G' else 1
        served_packets = min(self.queue, rate)
        self.queue -= served_packets
        return served_packets

    def end_slot(self):
        packet = np.random.binomial(n=1, p=self.p_packet)
        dropped = 0
        if self.queue == C:
            dropped = packet
        else:
            self.queue += packet
    
        if self.state == 'B':
            self.state = 'G' if np.random.binomial(n=1, p=self.p_bg) else 'B'
        else:
            self.state = 'B' if np.random.binomial(n=1, p=self.p_gb) else 'G'

        return dropped

In [ ]:
K = 2
C = 4
N = 2
states = tuple(itertools.product(range(C+1), ['B', 'G'], repeat=K))

#### Policies
In both policies, the tie-breaker rule is to just choose the first index

In [ ]:
def best_channel_policy(channels):
    for i, channel in enumerate(channels):
        if channel.state == 'G':
            return i
    return 0

def longest_queue_policy(channels):
    return np.argmax([channel.queue for channel in channels]) #argmax returns first index of max element if it encounters tie conditions

def best_channel_policy_sim(state):
    for i, channel in enumerate(state[1::2]):
        if channel == 'G':
            return i
    return 0

def longest_queue_policy_sim(state):
    queues = state[0::2]
    return np.argmax([queue for queue in queues]) #argmax returns first index of max element if it encounters tie conditions

In [ ]:
def solve_policy_evaluation(states, policy, p_bg, p_gb, p_packet, l, gamma):
    P, r, state_to_index = get_mdp_matrices(states, policy, p_bg, p_gb, p_packet, l)

    I = np.eye(len(states))

    V = np.linalg.solve(I - gamma * P, r)

    return V, P, r, state_to_index

def get_mdp_matrices(states, policy, p_bg, p_gb, p_packet, l):
    n = len(states)
    state_to_index = {s: i for i, s in enumerate(states)}

    P = np.zeros((n, n))
    r = np.zeros(n)

    for i, state in enumerate(states):

        action = policy(state)
        queue_sizes = state[0::2]

        channel = state[c_index(action)]
        rate = N if channel == 'G' else 1
        transmitted_packets = min(queue_sizes[action], rate)
        
        dropped_packets = 0

        for j in range(K):
            qj = queue_sizes[j]

            if j == action:
                qj = max(0, qj - transmitted_packets)

            if qj == C:
                dropped_packets += p_packet[j]

        r[i] = ((1-l) * transmitted_packets) - (l * dropped_packets)

        temp_state = list(state)
        qi = 2 * action
        temp_state[qi] = max(0, temp_state[qi] - transmitted_packets)
        base_state = tuple(temp_state)

        transitions = generate_transitions(base_state, p_bg, p_gb, p_packet)

        for next_state, prob in transitions:
            j = state_to_index[next_state]
            P[i, j] += prob

    return P, r, state_to_index

def generate_transitions(temp_state, p_bg, p_gb, p_packet):
    results = []

    for mask in range(1 << K):
        for arrival_mask in range(1 << K):
            s = list(temp_state)
            prob = 1.0
        
            for i in range(K):
                ci = c_index(i)
                c = temp_state[ci]
        
                flip_event = (mask >> i) & 1
        
                if flip_event:
                    s[ci] = flip(c)
                    prob *= p_gb[i] if c == 'G' else p_bg[i]
                else:
                    prob *= (1 - p_gb[i] if c == 'G' else 1 - p_bg[i])
        
            for i in range(K):
                qi = q_index(i)
        
                arrival = (arrival_mask >> i) & 1

                if arrival:
                    if s[qi] < C:
                        s[qi] += 1
                    prob *= p_packet[i]
                else:
                    prob *= (1 - p_packet[i])

            results.append((tuple(s), prob))

    return results

def q_index(i):
    return 2 * i

def c_index(i):
    return 2 * i + 1

def flip(c):
    return 'G' if c == 'B' else 'B'

In [ ]:
def monte_carlo_value(p_bg, p_gb, p_packet, policy, epochs, T, gamma, l):
    
    returns = []

    for _ in range(epochs):
        users = []
        for i in range(K):
            u = User(C, p_bg[i], p_gb[i], p_packet[i], 'B')
            users.append(u)
            
        G = run_episode(users, policy, T, gamma, l)
        returns.append(G)

    return np.mean(returns)

def run_episode(users, policy, T, gamma, l):
    total_return = 0
    discount = 1.0

    for t in range(T):

        action = policy(users)

        served = users[action].serve(N)

        dropped = 0
        for u in users:
            dropped += u.end_slot()

        r = ((1-l)*served) - (l*dropped)

        total_return += discount * r
        discount *= gamma

    return total_return

In [ ]:
np.random.seed(int(STUDENT_ID[-2:]))
p_bg = np.random.uniform(low=0.1, high=0.3, size=K)
p_gb = np.random.uniform(low=0.1, high=0.3, size=K)
p_packet= np.random.uniform(low=0.6, high=0.9, size=K)

l = np.random.uniform(low=0.2, high=0.8)

V_evaluation, _, _, state_to_index = solve_policy_evaluation(states, best_channel_policy_sim, p_bg, p_gb, p_packet, l, 0.35)
V_mc = monte_carlo_value(p_bg, p_gb, p_packet, best_channel_policy, 500, 20000, 0.35, l)

index = state_to_index[(0, 'B', 0, 'B')]

print(f"\nBest Channel policy with starting state {states[index]}:")
print(f"Policy evaluation: {V_evaluation[index]}")
print(f"Simulation: {V_mc}")
print(f"Error: {abs(V_mc - V_evaluation[index])}\n")

V_evaluation, _, _, state_to_index = solve_policy_evaluation(states, longest_queue_policy_sim, p_bg, p_gb, p_packet, l, 0.35)
V_mc = monte_carlo_value(p_bg, p_gb, p_packet, longest_queue_policy, 500, 20000, 0.35, l)

print(f"\nLongest queue policy with starting state {states[index]}:")
print(f"Policy evaluation: {V_evaluation[index]}")
print(f"Simulation: {V_mc}")
print(f"Error: {abs(V_mc - V_evaluation[index])}")

In [ ]:
def get_action_values(state, V, state_to_index, p_bg, p_gb, p_packet, l, gamma):
    action_values = []
    queue_sizes = state[0::2]

    for action in range(K):
        channel = state[c_index(action)]
        rate = N if channel == 'G' else 1
        transmitted_packets = min(queue_sizes[action], rate)

        dropped_packets = 0
        for j in range(K):
            qj = queue_sizes[j]
            if j == action:
                qj = max(0, qj - transmitted_packets)
            if qj == C:
                dropped_packets += p_packet[j]

        reward = ((1-l) * transmitted_packets - (l*dropped_packets))

        temp_state = list(state)
        qi = q_index(action)
        temp_state[qi] = max(0, temp_state[qi] - transmitted_packets)
        base_state = tuple(temp_state)

        transitions = generate_transitions(base_state, p_bg, p_gb, p_packet)

        expected_future_return = 0
        for next_state, prob in transitions:
            idx = state_to_index[next_state]
            expected_future_return += prob * V[idx]

        q_value = reward + (gamma * expected_future_return)
        action_values.append(q_value)

    return action_values

def value_iteration(states, p_bg, p_gb, p_packet, l, gamma, epsilon=1e-6):
    n = len(states)
    state_to_index = {s: i for i, s in enumerate(states)}
    V = np.zeros(n)
    action_values_table = []

    while True:
        delta = 0

        V_prev = np.copy(V)

        for i, state in enumerate(states):
            action_values = get_action_values(state, V_prev, state_to_index, p_bg, p_gb, p_packet, l, gamma)
            action_values_table.append(action_values)

            best_value = np.max(action_values)
            delta = max(delta, abs(best_value - V[i]))
            V[i] = best_value

        if delta < epsilon:
            break

    return V, action_values_table

def optimal_policy(states, action_values_table):
    policy = {}
    for i, state in enumerate(states):
        action_values = action_values_table[i]

        policy[state] = np.argmax(action_values).item()

    return policy

In [ ]:
np.random.seed(int(STUDENT_ID[-2:]))
p_bg = np.random.uniform(low=0.1, high=0.3, size=K)
p_gb = np.random.uniform(low=0.1, high=0.3, size=K)
p_packet= np.random.uniform(low=0.6, high=0.9, size=K)

l = np.random.uniform(low=0.2, high=0.8)

V_95, action_values_table = value_iteration(states, p_bg, p_gb, p_packet, l, 0.95)
policy_95 = optimal_policy(states, action_values_table)

#start_state = (0, 'B', 0, 'B')
#start_index = states.index(start_state)

#print(f"Optimal Value V* for state {start_state} and gamma 0.95: {V_95[start_index]:.6f}\n")

#Uncomment to see Policy table
#for s, a in policy_95.items(): print(f"State: {s} | Action: Serve User {a+1}")

V_0, action_values_table = value_iteration(states, p_bg, p_gb, p_packet, l, 0)
policy_0 = optimal_policy(states, action_values_table)

#print(f"Optimal Value V* for state {start_state} and gamma 0: {V_0[start_index]:.6f}")

#Uncomment to see Policy table
#for s, a in policy_0.items(): print(f"State: {s} | Action: Serve User {a+1}")

In [ ]:
def monte_carlo_metrics(p_bg, p_gb, p_packet, policy_table, epochs, T, gamma, l):
    all_returns = []
    all_throughput = []
    all_drops = []

    for _ in range(epochs):
        users = [User(C, p_bg[i], p_gb[i], p_packet[i], 'B') for i in range(K)]
        
        G, total_t, total_d = run_episode_table(users, policy_table, T, gamma, l)
        
        all_returns.append(G)
        all_throughput.append(total_t / T)
        all_drops.append(total_d / T)

    return np.mean(all_returns), np.mean(all_throughput), np.mean(all_drops)

def run_episode_table(users, policy_table, T, gamma, l):
    total_return = 0
    cum_throughput = 0
    cum_drops = 0
    discount = 1.0

    for t in range(T):
        current_state = []
        for u in users:
            current_state.append(u.queue)
            current_state.append(u.state)
        current_state = tuple(current_state)

        action = policy_table[current_state]

        served = users[action].serve(N)
        dropped = sum(u.end_slot() for u in users)

        reward = ((1-l) * served) - (l * dropped)
        
        total_return += discount * reward
        cum_throughput += served
        cum_drops += dropped
        discount *= gamma

    return total_return, cum_throughput, cum_drops

def create_baseline_table(states, policy_type):
    table = {}
    for state in states:
        if policy_type == 'best_channel':
            channels = state[1::2]
            best_user = 0
            for i, ch in enumerate(channels):
                if ch == 'G':
                    best_user = i
                    break
            table[state] = best_user
            
        elif policy_type == 'longest_queue':
            queues = state[0::2]
            table[state] = np.argmax(queues)
            
    return table

In [ ]:
bc_table = create_baseline_table(states, 'best_channel')
lq_table = create_baseline_table(states, 'longest_queue')


results = {}
for name, pol in [("Best Channel", bc_table), ("Longest Queue", lq_table), ("Optimal 0.95", policy_95), ("Optimal 0", policy_0)]:
    v, t, d = monte_carlo_metrics(p_bg, p_gb, p_packet, pol, 500, 20000, 0.95, l)
    results[name] = (v, t, d)

# 3. Clean Print
print(f"{'Policy':<20} | {'Avg Return':<12} | {'Througput/Slot':<15} | {'Drops/Slot':<12}")
print("-" * 65)
for name, (v, t, d) in results.items():
    print(f"{name:<20} | {v:<12.4f} | {t:<15.4f} | {d:<12.4f}")